# Part 4 — Scaling Up: BPE, AMP, LR Schedule, Checkpoints, Logging

**Goal of this notebook:** Walk through every ingredient that turns the Part 3 modern model into a real *training* run. By the end, every line of `train.py` should make sense.

**What you should know going in:**
- Parts 1-3 (Transformer block, modern attention, RMSNorm/RoPE/SwiGLU) are fresh
- Comfort with PyTorch tensors, optimizers, DataLoader
- (Optional) you've seen `tensorboard` once before

**What you'll have at the end:**
- A trained BPE tokenizer over a small sample text; intuition for vocab/merges
- A clear picture of the **(x, y) shifted-pair** training signal
- The **warmup + cosine** LR curve plotted, with the math behind both phases
- A working understanding of **AMP** (`GradScaler` + autocast) and **gradient accumulation**
- A round-trip **checkpoint** (model + optimizer + scheduler + scaler + step + config), including the architecture-verify guard
- A small **TensorBoard** logger demo (scalars / histograms / text)
- A complete *single training step* traced through every module, end to end

We use a small anchor text throughout:
> `"i love deep learning. i love coding. coding is deep learning."`


## The Map

The Part 4 training loop has ten parts arranged in a cycle. The master diagram below appears in every section with that section's node highlighted in green and the rest greyed out.

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#e8e8e8,stroke:#bbb,color:#777
```

**What's new vs Part 2's training loop**

| Part 2 (tiny) | Part 4 (production) | Section |
|---|---|---|
| byte tokenizer (vocab=256)            | **BPE** (vocab 8 k - 32 k)              | 4.1 |
| constant LR                           | **warmup + cosine** schedule            | 4.2 |
| fp32                                  | **AMP** with `GradScaler`               | 4.3 |
| batch fits in memory                  | **gradient accumulation**               | 4.3 |
| no resume                             | **checkpoint** model + opt + sched + scaler + step + config | 4.4 |
| print() loss                          | **TensorBoard** / W&B                   | 4.5 |
| straight loop                         | strict resume + SIGINT graceful save    | 4.6 |
| -                                     | **sample.py** from a saved checkpoint   | 4.7 |


## Setup

Run this cell once. It adds `part_4/` to the path so we can `import` from its modules and the Part 3 model.


In [ ]:
import sys, pathlib

# This notebook lives inside part_4/. Add it and part_3/ (for GPTModern) to sys.path.
NB_DIR = pathlib.Path().resolve()
PART_3 = NB_DIR.parent / "part_3"
for p in (str(NB_DIR), str(PART_3)):
    if p not in sys.path:
        sys.path.insert(0, p)

import math, time, json, tempfile, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

# The anchor text we reuse across sections
ANCHOR = "i love deep learning. i love coding. coding is deep learning."
print(f"Setup complete. Working directory: {NB_DIR}")
print(f"ANCHOR ({len(ANCHOR)} chars): {ANCHOR}")


---
## 4.1 BPE Tokenizer

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

In Part 2 / Part 3 we used a **byte tokenizer**: one ID per UTF-8 byte. Vocab = 256, easy to implement, but every word becomes 4 - 8 tokens. On a 256-token block, that's only ~30 - 50 English *words* of context.

**BPE** (Byte Pair Encoding) solves this by *learning* a vocabulary of frequent byte sequences. The algorithm is dead simple:

> Repeatedly find the most frequent adjacent token pair, merge it into a new token. Stop when the vocabulary reaches the target size.

The result: common words ("the", "love", "learning") get single IDs, rare words split into morpheme-like pieces. A trained 8 k BPE tokenizer typically compresses English text by ~3-5x vs byte level.

### Math + intuition

Start from the byte alphabet (256 tokens). At each step:

$$\text{pair}_t = \arg\max_{(a,b)} \#\{(a,b) \text{ adjacent in corpus}\}$$

Add a new token `ab` = merge of `a` and `b`, replace every occurrence in the corpus, repeat. Each merge becomes a rule stored in `merges.txt`. Saved artifacts:

| File | Role |
|---|---|
| `vocab.json` | `{token: id}` lookup |
| `merges.txt` | learned merge rules in order |
| `tokenizer.json` | full HF Tokenizer state |
| `bpe_meta.json` | `{vocab_size, special_tokens}` |

### Live demo — train BPE on our anchor text


In [ ]:
# Write the anchor text repeated 200x to a temp file so BPE has enough signal to find merges.
tmp = tempfile.mkdtemp()
data_path = pathlib.Path(tmp) / "anchor.txt"
data_path.write_text((ANCHOR + "\n") * 200, encoding="utf-8")
print(f"Wrote {data_path.stat().st_size} bytes of training text.")

# Train a tiny BPE
# NOTE: ByteLevelBPETokenizer starts with the 256-byte alphabet + special tokens (~261).
# vocab_size must exceed that to actually learn merges. 500 is plenty for our tiny demo.
try:
    from tokenizer_bpe import BPETokenizer
    tok = BPETokenizer(vocab_size=500)
    tok.train(str(data_path))
    tok_dir = pathlib.Path(tmp) / "tok"
    tok.save(str(tok_dir))
    print(f"\nSaved tokenizer to {tok_dir}")
    print("Files written:", sorted(p.name for p in tok_dir.iterdir()))
except ImportError as e:
    print("`tokenizers` package not installed. Run `pip install tokenizers` and re-run this cell.")
    raise


### Compare with byte tokenizer


In [ ]:
# Byte tokenizer (Part 2/3): one ID per UTF-8 byte
byte_ids = list(ANCHOR.encode("utf-8"))
bpe_ids  = tok.encode(ANCHOR)

print(f"text                 : {ANCHOR!r}")
print(f"byte tokenization    : {len(byte_ids):3d} ids  e.g. {byte_ids[:12]} ...")
print(f"BPE tokenization     : {len(bpe_ids):3d} ids  e.g. {bpe_ids[:12]} ...")
print(f"compression factor   : {len(byte_ids) / len(bpe_ids):.2f}x")
print()
print(f"BPE decode round-trip: {tok.decode(bpe_ids)!r}")


### Visualize the merges learned

Show the first 20 merge rules (read top-down — earlier merges are more frequent).


In [ ]:
merges_file = tok_dir / "merges.txt"
lines = merges_file.read_text(encoding="utf-8").splitlines()
print(f"Total merges learned: {len(lines)-1}")  # first line is a header
print()
print("First 20 merges (most frequent pairs):")
for i, line in enumerate(lines[1:21]):
    print(f"  {i+1:2d}. {line}")


**What you see:** the first merges combine very common pairs like `i n`, `e d`, `e r` -> single tokens (this is how words like "learning", "coding" become a single subword).

### Decode each BPE token individually

We can also see *what each token represents* by decoding one ID at a time.


In [ ]:
print(f"BPE tokens for: {ANCHOR!r}\n")
for tid in bpe_ids[:25]:
    piece = tok.decode([tid])
    print(f"  id {tid:4d}  ->  {piece!r}")


### Shape trace

| Stage | Shape / Type |
|---|---|
| `text` | `str` of length `N_chars` |
| `text.encode("utf-8")` | `bytes` of length `N_bytes ≥ N_chars` |
| `tok.encode(text)` | `list[int]` of length `N_tokens ≪ N_bytes` |
| `tok.decode(ids)` | `str` (round-trip) |

### Why BPE? (layered)

1. **Surface:** fewer tokens per sentence -> more context fits in `block_size`.
2. **Practical:** common subwords get their own embedding row -> the model learns "love" once instead of having to compose it from `[l, o, v, e]` four times.
3. **Deep:** information theory — BPE is a greedy approximation to a compression code over your corpus. Lower entropy per token means the model's softmax has an easier job.

**TL;DR:** BPE turns a 4-8x compression problem into an architectural freebie.


---
## 4.1 (continued) — Dataset and the (x, y) shift

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

Once the corpus is a long `LongTensor` of token IDs, how do we feed it to the model? **Causal LM training** wants pairs:

  * `x`: a length-`block_size` window of token IDs
  * `y`: the SAME window shifted by one position — i.e. the *next-token target* at every position

Position `t` in `x` should predict `y[t]`. Since the model is causal, position `t` only sees `x[0..t]`, so this is a *parallelizable* prediction task — one forward pass = `block_size` next-token predictions.

### Math — the shift

For a sentence with token IDs `[a, b, c, d, e, f]` and `block_size = 4`:

| t (position in window) | x[t] | y[t] |
|---|---|---|
| 0 | a | b |
| 1 | b | c |
| 2 | c | d |
| 3 | d | e |

Then `loss = mean( CE(logits[t], y[t]) for t in 0..block_size-1 )`. We just *throw away* the prediction at position `block_size-1` if it would need `f` (we just shift left by 1).

### Live demo


In [ ]:
from dataset_bpe import TextBPEBuffer, make_loader

block_size = 12
ds = TextBPEBuffer(str(data_path), tokenizer=tok, block_size=block_size)
print(f"Dataset length (#sliding windows): {len(ds)}")
print(f"First raw item: x, y of shape {ds[0][0].shape}, {ds[0][1].shape}")

x0, y0 = ds[0]
print()
print("x[0] (window of IDs)        :", x0.tolist())
print("y[0] = x[0] shifted by +1   :", y0.tolist())
print()
print("Decoded x:", repr(tok.decode(x0.tolist())))
print("Decoded y:", repr(tok.decode(y0.tolist())))


### DataLoader: batches of windows


In [ ]:
loader = make_loader(str(data_path), tok, block_size=block_size, batch_size=4, shuffle=True)
xb, yb = next(iter(loader))
print(f"Batch x shape: {tuple(xb.shape)}   (batch_size, block_size)")
print(f"Batch y shape: {tuple(yb.shape)}   (batch_size, block_size)")
print(f"y == x shifted by 1? -> Compare row 0:")
print("  x[0]:", xb[0].tolist())
print("  y[0]:", yb[0].tolist())


---
## 4.2 Warmup + Cosine LR Schedule

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

Why not just use a constant learning rate? Two reasons:

1. **At the start**, the model's weights are random and Q/K dot products are wild. A big LR step here can blow up the loss or undo the careful init.
2. **At the end**, you want fine adjustments, not large noisy ones. Decaying the LR gives the model a chance to settle into a sharp minimum.

The standard LLM recipe combines:
  * **Linear warmup** for the first `warmup_steps` steps (LR climbs from 0 to `base_lr`)
  * **Cosine decay** thereafter, smoothly to 0 over the remaining steps

### Math

$$\text{lr}(t) = \begin{cases} \text{base\_lr} \cdot \dfrac{t}{\text{warmup}} & \text{if } t \le \text{warmup} \\[4pt] \tfrac{1}{2}\,\text{base\_lr}\,\big(1 + \cos\big(\pi \cdot \tfrac{t - \text{warmup}}{\text{total} - \text{warmup}}\big)\big) & \text{if } t > \text{warmup} \end{cases}$$

### Visualization: plot the schedule


In [ ]:
from lr_scheduler import WarmupCosineLR

class DummyOpt:
    def __init__(self, lr=0.0):
        self.param_groups = [{"lr": lr}]

base_lr = 3e-4
total = 1000
warmup = 100

opt = DummyOpt(0.0)
sched = WarmupCosineLR(opt, warmup_steps=warmup, total_steps=total, base_lr=base_lr)
lrs = [sched.step() for _ in range(total)]

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(lrs, lw=2, color="#28a745")
ax.axvspan(0, warmup, alpha=0.15, color="orange", label=f"linear warmup (first {warmup} steps)")
ax.axhline(base_lr, ls="--", lw=0.6, color="grey", label=f"base_lr = {base_lr:.0e}")
ax.set_xlabel("optimizer step")
ax.set_ylabel("learning rate")
ax.set_title(f"WarmupCosineLR: warmup={warmup}, total={total}, base_lr={base_lr:.0e}")
ax.legend(); ax.grid(alpha=0.3); plt.show()

print(f"lr[0]            = {lrs[0]:.2e}    (just stepped from 0)")
print(f"lr[warmup]       = {lrs[warmup-1]:.2e}    (peak, end of warmup)")
print(f"lr[mid]          = {lrs[total//2]:.2e}")
print(f"lr[end]          = {lrs[-1]:.2e}    (near zero)")


### Hand-traced cells

At `step = 50` (mid-warmup, `warmup = 100`):

$$\text{lr}(50) = 3 \times 10^{-4} \cdot \frac{50}{100} = 1.5 \times 10^{-4}$$

At `step = 550` (halfway through decay, `(550 - 100) / (1000 - 100) = 0.5`):

$$\text{lr}(550) = \tfrac{1}{2} \cdot 3 \times 10^{-4} \cdot (1 + \cos(\pi \cdot 0.5)) = 1.5 \times 10^{-4}$$

(The midpoint of decay sits at half the peak — that's the cosine's symmetry.)


In [ ]:
print("step  50 lr =", lrs[49],   "   manual:", base_lr * 50/warmup)
print("step 550 lr =", lrs[549],  "   manual:", 0.5 * base_lr * (1 + math.cos(math.pi * 0.5)))


---
## 4.3 AMP (mixed precision) + Gradient Accumulation

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#e8e8e8,stroke:#bbb,color:#777
```

These are two *independent* tricks that `AmpGrad` happens to bundle together.

### Part A — Mixed precision (AMP)

#### The question

fp16 is ~2x faster and uses half the memory of fp32 on every NVIDIA GPU since Volta (V100). The catch: fp16's exponent range is small (~6e-5 to ~6e4), so **gradients underflow to zero** on small magnitudes.

#### The trick: GradScaler

Multiply the loss by a large factor `S` *before* `.backward()`. The chain rule scales every gradient by `S` too, pushing them up into fp16's safe range. Just before `optimizer.step()`, divide the gradients by `S` to restore the correct magnitudes.

If any gradient overflows to `inf` or `nan`, skip this step entirely and *halve* `S` next time. If we go many steps without overflow, *double* `S`. That's `GradScaler` in three sentences.

#### Visualization: fp16's range and the underflow problem


In [ ]:
# Show what happens to a tiny gradient when stored in fp16
small_grad = torch.tensor(1e-7, dtype=torch.float32)
small_grad_fp16 = small_grad.half()
print(f"fp32  : {small_grad.item():.3e}")
print(f"fp16  : {small_grad_fp16.item():.3e}   <- underflowed to 0!")
print()

scale = 1024.0
scaled = (small_grad * scale).half()
unscaled = scaled.float() / scale
print(f"With scaler (S=1024):")
print(f"  scale loss  -> {scaled.item():.3e}    (survives fp16)")
print(f"  unscale grad -> {unscaled.item():.3e}  (recovered original)")


### Part B — Gradient accumulation

#### The question

You want an effective batch of 64 sequences, but only 16 fit in VRAM. Run **4 micro-batches of 16**, calling `.backward()` after each one *without zeroing gradients in between*. PyTorch accumulates them, so after 4 calls the gradient buffer holds the sum from all 64 sequences. Then step + zero.

#### Math

For a loss averaged over the batch:

$$L_{\text{batch}} = \frac{1}{N} \sum_{i=1}^{N} L_i, \quad \nabla L_{\text{batch}} = \frac{1}{N} \sum_i \nabla L_i$$

If we split into `accum` micro-batches of size `N / accum`:

$$L_{\text{micro}} = \frac{1}{N/\text{accum}} \sum_{i \in \text{micro}} L_i$$

Naively summing gradients across micro-batches would overcount by `accum`. The fix is in `AmpGrad.backward()`:

```python
loss = loss / self.accum   # rescale so the accumulated sum matches the big-batch gradient
```

#### Live demo — same effective gradient, different memory usage


In [ ]:
from amp_accum import AmpGrad

torch.manual_seed(0)
# Two tiny linear models with the same init
model_big   = nn.Linear(8, 4)
model_accum = nn.Linear(8, 4)
model_accum.load_state_dict(model_big.state_dict())

# One big batch of 16
x_big = torch.randn(16, 8); y_big = torch.randn(16, 4)
loss_big = F.mse_loss(model_big(x_big), y_big)
loss_big.backward()
grad_big = model_big.weight.grad.clone()

# Four micro-batches of 4
opt_a = torch.optim.SGD(model_accum.parameters(), lr=0.0)
amp = AmpGrad(opt_a, accum=4, amp=False)
for k in range(4):
    xs = x_big[k*4:(k+1)*4]; ys = y_big[k*4:(k+1)*4]
    loss = F.mse_loss(model_accum(xs), ys)
    amp.backward(loss)
grad_accum = model_accum.weight.grad.clone()

print(f"max abs diff between gradients: {(grad_big - grad_accum).abs().max().item():.3e}")
print("(They should match to ~1e-7 — accumulated micro-batches reproduce the big-batch gradient.)")


### The `AmpGrad` interface

```python
amp = AmpGrad(optimizer, accum=4, amp=True)

# inside the training loop:
with torch.cuda.amp.autocast(enabled=amp.amp):
    logits, loss, _ = model(xb, yb)
amp.backward(loss)            # scales + divides by accum + accumulates

if amp.should_step():         # every accum micro-batches
    amp.step()                # scaler.step + scaler.update
    amp.zero_grad()
    lr = scheduler.step()
```


---
## 4.4 Checkpointing

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
```

### The question

A real training run lasts hours or days. You need to:
- save the model at regular intervals
- resume *exactly* where you left off — same step, same LR, same scaler state
- catch the classic mistake of loading weights into the *wrong architecture* before silent gibberish happens

### What goes in `model_last.pt`

| Key | Why |
|---|---|
| `model` | weights |
| `optimizer` | momentum / variance state for AdamW |
| `scheduler` | step counter (so LR resumes mid-decay) |
| `amp_scaler` | dynamic scale, so AMP doesn't have to re-tune from scratch |
| `step` | global optimizer step (for logger / checkpoint counters) |
| `config` | model dimensions — used to refuse loads with mismatched architecture |
| `version` | bookkeeping |

A sibling file `tokenizer_dir.txt` records the path of the BPE tokenizer used, so `sample.py` and resumed `train.py` can find it.

### Live demo — round-trip a tiny model


In [ ]:
from checkpointing import save_checkpoint, load_checkpoint
from lr_scheduler import WarmupCosineLR
from amp_accum import AmpGrad
from model_modern import GPTModern

# Build a tiny GPTModern
def build_tiny():
    return GPTModern(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=4, n_embd=64, n_kv_head=2, use_rmsnorm=True,
                     use_swiglu=True, rope=True, max_pos=128)

torch.manual_seed(0)
m   = build_tiny()
opt = torch.optim.AdamW(m.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
sch = WarmupCosineLR(opt, warmup_steps=10, total_steps=100, base_lr=3e-4)
amp = AmpGrad(opt, accum=2, amp=False)

# Pretend we trained a few steps
for _ in range(7):
    sch.step()
fake_step = 7

# Save
out_dir = pathlib.Path(tmp) / "ckpt"
save_checkpoint(m, opt, sch, amp, step=fake_step, out_dir=str(out_dir),
                tokenizer_dir=str(tok_dir), config=None)
print("Files in checkpoint dir:")
for p in sorted(out_dir.iterdir()):
    print(f"  {p.name}   ({p.stat().st_size:,} bytes)")


### Inspect what's inside


In [ ]:
ckpt = torch.load(out_dir / "model_last.pt", map_location="cpu", weights_only=False)
print("Top-level keys in model_last.pt:")
for k, v in ckpt.items():
    if k == "model":
        print(f"  {k:12s} : dict with {len(v)} tensors (state_dict)")
    elif k == "config":
        print(f"  {k:12s} : {v}")
    elif isinstance(v, dict):
        print(f"  {k:12s} : dict with keys {list(v.keys())}")
    else:
        print(f"  {k:12s} : {v!r}")


### Round-trip: load into a fresh model


In [ ]:
# Build a fresh model with the SAME architecture and load
m2 = build_tiny()
opt2 = torch.optim.AdamW(m2.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
sch2 = WarmupCosineLR(opt2, warmup_steps=10, total_steps=100, base_lr=3e-4)
amp2 = AmpGrad(opt2, accum=2, amp=False)

restored_step = load_checkpoint(m2, str(out_dir / "model_last.pt"),
                                optimizer=opt2, scheduler=sch2, amp=amp2, strict=True)
print(f"Restored step: {restored_step}")

# Check weights match exactly
all_match = all(torch.equal(p1, p2) for p1, p2 in zip(m.parameters(), m2.parameters()))
print(f"All parameters identical after round-trip? {all_match}")


### The architecture-verify guard

If we try to load the checkpoint into a model with *different* dimensions, we want a clear error, not silent weight-shape papering. Try loading a 4-layer model from a 2-layer ckpt:


In [ ]:
# Build a model with a mismatched architecture (n_layer=4 instead of 2)
m_wrong = GPTModern(vocab_size=tok.vocab_size, block_size=32, n_layer=4,   # <-- changed
                    n_head=4, n_embd=64, n_kv_head=2, use_rmsnorm=True,
                    use_swiglu=True, rope=True, max_pos=128)

try:
    load_checkpoint(m_wrong, str(out_dir / "model_last.pt"), strict=True)
    print("(unexpected) load succeeded")
except RuntimeError as e:
    print("PASS - raised:", str(e).splitlines()[0])
    for line in str(e).splitlines()[1:5]:
        print("       ", line)


---
## 4.5 Logging — NoopLogger / TBLogger / WBLogger

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ckpt fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

You want to track `loss`, `lr`, gradient norm, throughput, sample text, parameter histograms — all on a smooth dashboard you can inspect during long runs. The standard answer is **TensorBoard** (or W&B). Part 4 ships three interchangeable backends behind the same `.log(step=..., **kv)` contract.

### Three backends, one interface

```python
class NoopLogger:    def log(self, **kv): pass
class TBLogger:      # SummaryWriter under the hood
class WBLogger:      # wandb.log under the hood
```

The training loop calls `logger.log(step=step, loss=..., lr=..., ...)` and is happy whether the backend is real or a no-op.

### Auto-routing inside `TBLogger.log()`

| Value | Becomes |
|---|---|
| `key` starts with `text/` | `add_text(key[5:], str(value))` |
| 1-element tensor/ndarray  | `add_scalar(key, float(value))` |
| small tensor (<= 2048 el) | `add_histogram(key, value)` |
| large tensor              | `add_scalar(key/mean), add_scalar(key/std)` |
| Python number             | `add_scalar(key, float(value))` |

This lets us log scalars and distributions through the same call.

### Live demo (TB writer in a temp dir)


In [ ]:
from logger import init_logger, NoopLogger, TBLogger

log_dir = pathlib.Path(tmp) / "tblogs"
log_dir.mkdir(parents=True, exist_ok=True)

# Try TBLogger; fall back to NoopLogger if tensorboard isn't available
logger = init_logger("tensorboard", out_dir=str(log_dir))
print(f"Logger type: {type(logger).__name__}")

# Log a few synthetic values
for step in range(5):
    logger.log(
        step=step,
        loss=2.5 * 0.9 ** step,           # scalar
        lr=3e-4 * step / 4,                # scalar
        **{"sys/throughput_tokens_per_s": 1000.0 + 50 * step},
    )
    # log a histogram via the hist helper
    if hasattr(logger, "hist"):
        logger.hist("debug/random_tensor", torch.randn(256), step)
    # log a sample text
    if hasattr(logger, "text"):
        logger.text("samples/dummy", f"step {step}: hello world", step)

if hasattr(logger, "flush"):
    logger.flush()

if hasattr(logger, "run_dir"):
    print(f"Wrote TB events under: {logger.run_dir}")
    print("Inspect with:  tensorboard --logdir", str(log_dir))


**To view it locally:**

```bash
tensorboard --logdir <log_dir>
```

Then open `http://localhost:6006/`.


---
## 4.6 Training Loop — single step traced end-to-end

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style bpe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ds fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style fwd fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style loss fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style amp fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style opt fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style sched fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style log fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ckpt fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
```

Now we glue everything together. Below is a **single training step** that exercises every module from sections 4.1 - 4.5 in the same order `train.py` does.

### The forward / backward / step pattern


In [ ]:
# Build a tiny model + optimizer + scheduler + amp + logger from scratch.
torch.manual_seed(0)
device = torch.device("cpu")   # works everywhere; train.py picks cuda if available
model = build_tiny().to(device)
optim = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
sched = WarmupCosineLR(optim, warmup_steps=10, total_steps=100, base_lr=3e-4)
amp   = AmpGrad(optim, accum=2, amp=False)    # CPU here -> turn AMP off
logger = NoopLogger()                          # silent for the walkthrough

train_loader = make_loader(str(data_path), tok, block_size=block_size, batch_size=4, shuffle=True)
print(f"Model params      : {sum(p.numel() for p in model.parameters()):,}")
print(f"Loader length     : {len(train_loader)} batches")


In [ ]:
model.train()
step = 0
losses = []

# Two micro-batches per optimizer step (accum=2)
for micro, (xb, yb) in enumerate(train_loader):
    xb, yb = xb.to(device), yb.to(device)

    # 1) forward + CE loss
    logits, loss, _ = model(xb, yb)
    losses.append(loss.item())

    # 2) scaled backward + accumulate
    amp.backward(loss)

    print(f"micro {micro}: loss={loss.item():.4f}  should_step={amp.should_step()}")

    # 3) every `accum` micro-batches, take a real optimizer step
    if amp.should_step():
        amp.step()
        amp.zero_grad()
        lr = sched.step()
        step += 1
        logger.log(step=step, loss=loss.item(), lr=lr)
        print(f"  -> stepped: step={step}, lr={lr:.6f}")

    if step >= 2:
        break


### What just happened — line-by-line

| Block | What it did |
|---|---|
| `make_loader(...)` | tokenized the file once, wrapped it in DataLoader with `batch_size=4`, `block_size=12` |
| `xb, yb = batch` | both shape `(4, 12)` — `yb` is `xb` shifted by one position |
| `model(xb, yb)` | forward through GPTModern (Part 3), returns logits and CE loss |
| `amp.backward(loss)` | divides loss by `accum=2`, calls `loss.backward()` (or scaled if AMP on) |
| `amp.should_step()` | True every 2 calls — the accumulation boundary |
| `amp.step()` | optimizer.step() (under the GradScaler if AMP) |
| `amp.zero_grad()` | clear gradient buffers for the next accumulation cycle |
| `sched.step()` | advance LR (warmup or cosine) and write it into `optim.param_groups[i]['lr']` |
| `logger.log(...)` | TB/W&B if configured, else no-op |

The complete `train.py` adds:
  * the outer `while step < args.steps` driving the loop
  * periodic `atomic_save_all(...)` every `save_every` steps
  * SIGINT/SIGTERM handler that triggers one last save and exits cleanly
  * `init_logger`, `_log_hparams_tb`, `_maybe_log_graph_tb`, runtime + QKV histogram logging


---
## 4.7 Sample — load a checkpoint and generate

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#e8e8e8,stroke:#bbb,color:#777
    style bpe fill:#e8e8e8,stroke:#bbb,color:#777
    style ds fill:#e8e8e8,stroke:#bbb,color:#777
    style fwd fill:#e8e8e8,stroke:#bbb,color:#777
    style loss fill:#e8e8e8,stroke:#bbb,color:#777
    style amp fill:#e8e8e8,stroke:#bbb,color:#777
    style opt fill:#e8e8e8,stroke:#bbb,color:#777
    style sched fill:#e8e8e8,stroke:#bbb,color:#777
    style log fill:#e8e8e8,stroke:#bbb,color:#777
    style ckpt fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
```

`sample.py` is the inverse of `train.py`: read `model_last.pt`, reconstruct the model from the saved `config`, load the tokenizer from the `tokenizer_dir.txt` sidecar, and call `model.generate(...)`.

We've already exercised both pieces above (round-trip in 4.4, generation in Part 3's section 3.7). The script itself is 80 lines and just chains them:

```python
ckpt = torch.load(args.ckpt, map_location='cpu')
cfg = ckpt.get('config')
model = GPTModern(**cfg).eval()
model.load_state_dict(ckpt['model'])

tok = BPETokenizer(); tok.load(<path from tokenizer_dir.txt>)

ids = tok.encode(args.prompt) or [10]
out = model.generate(torch.tensor([ids]), max_new_tokens=args.tokens)
print(tok.decode(out[0].tolist()))
```

### Smoke generation from our checkpoint


In [ ]:
# Use the checkpoint we saved in 4.4
ckpt_path = out_dir / "model_last.pt"
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)

m_sample = GPTModern(**ck['config']).eval()
m_sample.load_state_dict(ck['model'])

# Encode the anchor sentence beginning as a prompt
prompt_ids = tok.encode("i love")
print(f"Prompt: 'i love'  -> ids = {prompt_ids}")
prompt = torch.tensor([prompt_ids], dtype=torch.long)

with torch.no_grad():
    out = m_sample.generate(prompt, max_new_tokens=30, temperature=1.0, top_k=50)
ids = out[0].tolist()
print(f"\nGenerated ids ({len(ids)}): {ids}")
print(f"Decoded text   : {tok.decode(ids)!r}")


(The text is garbage because the model is untrained — that's fine, we're verifying the pipeline shape, not quality.)

### Cleanup


In [ ]:
# Clean up the tempdir
try:
    shutil.rmtree(tmp)
    print(f"Removed temp dir {tmp}")
except Exception as e:
    print(f"(could not delete tempdir: {e})")


---
## Closing the loop

```mermaid
graph TD
    raw["raw text file<br>(tiny.txt)"]
    bpe["4.1 BPE Tokenizer<br>train + encode/decode"]
    ds["4.1 Dataset / DataLoader<br>(x, y) shifted batches"]
    fwd["forward: GPTModern (Part 3)"]
    loss["CE loss"]
    amp["4.3 AMP + grad accum<br>scaled backward"]
    opt["AdamW step<br>(β = 0.9, 0.95; wd = 0.1)"]
    sched["4.2 Warmup + Cosine LR"]
    log["4.5 Logger<br>(TB / wandb / noop)"]
    ckpt["4.4 Checkpoint<br>model + opt + sched + scaler + step + cfg"]

    raw --> bpe --> ds --> fwd --> loss --> amp --> opt --> sched --> log --> ckpt
    ckpt -. next batch .-> fwd

    style raw fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style bpe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ds fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style fwd fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style loss fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style amp fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style opt fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style sched fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style log fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ckpt fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
```

You now have everything needed to read every file in `part_4/`:

- [tokenizer_bpe.py](tokenizer_bpe.py) — section 4.1
- [dataset_bpe.py](dataset_bpe.py) — section 4.1
- [lr_scheduler.py](lr_scheduler.py) — section 4.2
- [amp_accum.py](amp_accum.py) — section 4.3
- [checkpointing.py](checkpointing.py) — section 4.4
- [logger.py](logger.py) — section 4.5
- [train.py](train.py) — section 4.6 (the full loop with SIGINT, atomic checkpointing, hparam logging)
- [sample.py](sample.py) — section 4.7

### What's next

Part 5 introduces a **Mixture of Experts** block — same training recipe, hybrid dense + MoE blocks via a learned gating function. Parts 6 (SFT) / 7 (Reward Model) / 8 (PPO) / 9 (GRPO) all build on a **trained Part 4 BPE tokenizer + base checkpoint at `runs/part4-demo/`**, so run

    cd part_4
    python orchestrator.py --demo

at some point before exploring those later parts.
